# V18 – Kryptografie Teil 2: Praxis mit `hashlib`

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- mit `hashlib` MD5- und SHA-256-Hashes von Strings berechnen und als Hexadezimal-String ausgeben,
- erklären, warum vor dem Hashen immer ein `.encode("utf-8")` nötig ist,
- eine einfache Passwort-Vergleichs-Funktion implementieren,
- einen Mini-Wörterbuch-Angriff auf einen gegebenen SHA-256-Hash durchführen.

## ⏱️ Zeitbudget
Ca. 25 Minuten.

## 🧭 TL;DR
`hashlib.sha256(b"...").hexdigest()` liefert dir den SHA-256-Hash als 64 Zeichen langen Hexadezimal-String. Der Input muss vom Typ `bytes` sein – deshalb immer vorher `.encode("utf-8")` auf Strings anwenden. Zum Vergleichen zweier Hashes reicht ein schlichtes `==`.

## 📦 Voraussetzungen
- `00_python_recap.ipynb` (bytes vs. str, Listen-Lookup)
- `01_theorie.ipynb` (Hash-Eigenschaften, Wörterbuch-Angriff)

## 1. `hashlib` importieren

Das Modul **`hashlib`** ist Teil der Python-Standardbibliothek – du brauchst nichts nachzuinstallieren. Es enthält unter anderem die Hash-Funktionen `md5`, `sha1`, `sha256`, `sha512` und weitere.

In [1]:
import hashlib

print("Verfügbare garantierte Algorithmen:")
print(sorted(hashlib.algorithms_guaranteed))

Verfügbare garantierte Algorithmen:
['blake2b', 'blake2s', 'md5', 'sha1', 'sha224', 'sha256', 'sha384', 'sha3_224', 'sha3_256', 'sha3_384', 'sha3_512', 'sha512', 'shake_128', 'shake_256']


## 2. Der erste MD5-Hash

Der einfachste Aufruf sieht so aus: Du übergibst `hashlib.md5(...)` die Eingabe als `bytes`-Objekt und hängst `.hexdigest()` an. Das Ergebnis ist ein **Hexadezimal-String** – bei MD5 immer exakt 32 Zeichen lang, was 128 Bit entspricht.

In [2]:
import hashlib

hash_md5 = hashlib.md5(b"hallo").hexdigest()
print(hash_md5)
print("Länge:", len(hash_md5), "Hex-Zeichen")

598d4c200461b81522a3328565c25f7c
Länge: 32 Hex-Zeichen


Der erwartete Output ist `598d4c200461b81522a3328565c25f7c`. Jede Python-Installation auf jedem Betriebssystem liefert denselben Wert – daran erkennst du die **deterministische** Eigenschaft aus der Theorie.

## 3. SHA-256 – der moderne Standard

Der Aufruf ist völlig analog, nur dass du `hashlib.sha256` statt `hashlib.md5` verwendest. Der Hexadezimal-Output ist dafür **doppelt so lang** (64 Zeichen = 256 Bit).

In [3]:
import hashlib

hash_sha = hashlib.sha256(b"test").hexdigest()
print(hash_sha)
print("Länge:", len(hash_sha), "Hex-Zeichen")

9f86d081884c7d659a2feaa0c55ad015a3bf4f1b2b0b822cd15d6c15b0f00a08
Länge: 64 Hex-Zeichen


Für `b"test"` ist der SHA-256-Hash `9f86d081884c7d659a2feaa0c55ad015a3bf4f1b2b0b822cd15d6c15b0f00a08`. Im Gegensatz zum 32-Zeichen-MD5-Wert ist das deutlich mehr Sicherheits-Spielraum gegen Brute-Force.

## 4. `.encode()` nicht vergessen

Ein häufiger Anfänger-Fehler: Man übergibt einen normalen `str` statt `bytes`. Python wirft dann einen `TypeError`, denn eine Hash-Funktion weiß nicht, welche Kodierung du möchtest. Führe das nächste Beispiel aus, um den Fehler zu sehen – der `try/except`-Block fängt ihn ab.

In [4]:
import hashlib
try:
    hashlib.sha256("test").hexdigest()   # str, nicht bytes!
except TypeError as e:
    print("TypeError:", e)

TypeError: Strings must be encoded before hashing


Die korrekte Form ist `hashlib.sha256("test".encode("utf-8")).hexdigest()`. Die beiden folgenden Zeilen sind äquivalent.

In [5]:
import hashlib

h1 = hashlib.sha256(b"test").hexdigest()
h2 = hashlib.sha256("test".encode("utf-8")).hexdigest()
print(h1)
print(h2)
print("gleich?", h1 == h2)

9f86d081884c7d659a2feaa0c55ad015a3bf4f1b2b0b822cd15d6c15b0f00a08
9f86d081884c7d659a2feaa0c55ad015a3bf4f1b2b0b822cd15d6c15b0f00a08
gleich? True


## 5. Lawineneffekt live

Wir ändern die Eingabe um **ein einziges Zeichen** und beobachten, wie sich der Hash komplett umstellt.

In [6]:
import hashlib

for wort in ["passwort", "passwort!", "Passwort", "passwort "]:
    h = hashlib.sha256(wort.encode("utf-8")).hexdigest()
    print(f"{wort!r:15s} -> {h}")

'passwort'      -> 33c5ebbb01d608c254b3b12413bdb03e46c12797e591770ccf20f5e2819929b2
'passwort!'     -> 824761dcfabc5a3247e2e9b8f6602455f9becd8abeb5cfd4be36d1f0dad9cfba
'Passwort'      -> a36c101570cc4410993de5385ad7034adb2dae6a05139ac7672577803084634d
'passwort '     -> 9d0b79f14ea119d2dbc667ada140c3e1c543fd62c907bf76e404a754721cdf03


Obwohl die Eingaben sehr ähnlich aussehen, haben die Hashes optisch **nichts** gemeinsam. Das ist der Lawineneffekt aus dem Theorie-Notebook in Aktion.

## 6. Fill-in: MD5 von „geheim"

Baue den MD5-Hash des Strings `"geheim"` und speichere das Ergebnis als Hex-String in `ergebnis`.

In [7]:
import hashlib
# TODO: Berechne den MD5-Hash von "geheim" (als bytes!) und weise ihn ergebnis zu.
ergebnis = ""  # hier anpassen

In [8]:
# ▶️ Selbstkontrolle
try:
    erwartet = hashlib.md5(b"geheim").hexdigest()
    assert ergebnis == erwartet, f"Erwartet {erwartet}, bekommen {ergebnis!r}"
    print("✅ Richtig!")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Variable `ergebnis` existiert noch nicht.")

❌ Noch nicht ganz: Erwartet e8636ea013e682faf61f56ce1cb1ab5c, bekommen ''


## 7. Passwort-Check-Funktion

Ein realistischer Anwendungsfall: Der Server hat in der Datenbank den SHA-256-Hash eines Passworts gespeichert. Beim Login gibt der Nutzer seine Eingabe ein, und wir müssen prüfen, ob sie zum gespeicherten Hash passt.

Die Funktion ist nur drei Zeilen lang: Eingabe hashen, vergleichen, Boolean zurückgeben.

In [9]:
import hashlib

def passwort_korrekt(eingabe: str, gespeicherter_hash: str) -> bool:
    hash_eingabe = hashlib.sha256(eingabe.encode("utf-8")).hexdigest()
    return hash_eingabe == gespeicherter_hash

# Annahme: In der Datenbank steht der SHA-256-Hash von "s3cret!"
gespeicherter_hash = hashlib.sha256(b"s3cret!").hexdigest()

print(passwort_korrekt("s3cret!", gespeicherter_hash))   # True
print(passwort_korrekt("falsch", gespeicherter_hash))    # False
print(passwort_korrekt("S3cret!", gespeicherter_hash))   # False (Gross-/Kleinschreibung!)

True
False
False


## 8. Fill-in: Passwort-Check

Implementiere die Funktion `meine_pruefung(eingabe, hash_wert)`, die genau wie oben `True` liefert, wenn der SHA-256-Hash der Eingabe dem `hash_wert` entspricht.

In [10]:
import hashlib

def meine_pruefung(eingabe, hash_wert):
    # TODO: Hashe die eingabe mit SHA-256 und vergleiche mit hash_wert
    return False

In [11]:
# ▶️ Selbstkontrolle
try:
    h = hashlib.sha256(b"robot").hexdigest()
    assert meine_pruefung("robot", h) is True, "korrekte Eingabe sollte True liefern"
    assert meine_pruefung("falsch", h) is False, "falsche Eingabe sollte False liefern"
    print("✅ Richtig!")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Funktion `meine_pruefung` existiert noch nicht.")

❌ Noch nicht ganz: korrekte Eingabe sollte True liefern


## 9. Fill-in: Hash-Länge prüfen

Speichere in der Variablen `laenge_sha256` die Anzahl der **Hex-Zeichen** eines SHA-256-Hashes. Tipp: Berechne einen beliebigen SHA-256-Hash und nimm `len(...)`.

In [12]:
import hashlib

# TODO: Ermittle die Hex-Länge eines SHA-256-Hashes
laenge_sha256 = 0  # hier anpassen

In [13]:
# ▶️ Selbstkontrolle
try:
    assert laenge_sha256 == 64, f"SHA-256 hat 64 Hex-Zeichen, du hast: {laenge_sha256}"
    print("✅ Richtig – 64 Hex-Zeichen = 256 Bit.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Variable `laenge_sha256` existiert noch nicht.")

❌ Noch nicht ganz: SHA-256 hat 64 Hex-Zeichen, du hast: 0


## 10. Mini-Challenge: Wörterbuch-Angriff

Gegeben ist der folgende SHA-256-Hash. Er ist der Hash eines extrem unsicheren Passworts. Nutze die Wortliste `["test", "1234", "admin", "passwort"]` und finde heraus, welcher Eintrag passt. Speichere dein Ergebnis in der Variablen `knackpasswort`.

In [14]:
import hashlib

ziel_hash = hashlib.sha256(b"admin").hexdigest()   # Der zu knackende Hash
wortliste = ["test", "1234", "admin", "passwort"]

# TODO: Hashe jedes Wort aus der Wortliste und vergleiche mit ziel_hash.
#       Sobald du eine Übereinstimmung findest, weise das Wort an knackpasswort zu.
knackpasswort = None

In [15]:
# ▶️ Selbstkontrolle
try:
    assert knackpasswort == "admin", f"Erwartet 'admin', bekommen: {knackpasswort!r}"
    print("✅ Richtig – Passwort geknackt. Genau das ist ein Wörterbuch-Angriff.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Variable `knackpasswort` existiert noch nicht.")

❌ Noch nicht ganz: Erwartet 'admin', bekommen: None


## ✅ Zusammenfassung

| Aufgabe | Idiom |
|---|---|
| MD5 eines Strings | `hashlib.md5(s.encode("utf-8")).hexdigest()` |
| SHA-256 eines Strings | `hashlib.sha256(s.encode("utf-8")).hexdigest()` |
| Hashes vergleichen | einfach mit `==` |
| Passwort prüfen | Eingabe hashen, mit gespeichertem Hash vergleichen |
| Wörterbuch-Angriff | Wortliste durchlaufen, hashen, vergleichen |

## ➡️ Nächster Schritt
Jetzt bist du für die fünf Übungsaufgaben in [03_aufgaben.ipynb](03_aufgaben.ipynb) bereit.